In [1]:
import cv2
import math
import numpy as np
import os
import tkinter as tk
from tkinter import filedialog
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from ultralytics import YOLO

# ==========================================
# 1. 資料結構定義
# ==========================================
@dataclass
class Point2D:
    x: float
    y: float

@dataclass
class PlayerSkeleton:
    player_id: int
    hip: Point2D
    left_wrist: Point2D
    right_wrist: Point2D
    left_ankle: Point2D
    right_ankle: Point2D
    left_shoulder: Point2D
    right_shoulder: Point2D

# ==========================================
# 2. 籃球幾何與物理特徵計算引擎 (已加入防守者為 None 的安全機制)
# ==========================================
class UltimateBasketballEngine:
    @staticmethod
    def get_dist(p1: Point2D, p2: Point2D) -> float:
        return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

    @classmethod
    def analyze_real_analytics(cls, ball: Point2D, offender: PlayerSkeleton, defender: Optional[PlayerSkeleton]) -> Dict[str, any]:
        body_center_x = (offender.hip.x + offender.left_shoulder.x + offender.right_shoulder.x) / 3.0
        body_center_y = (offender.hip.y + offender.left_shoulder.y + offender.right_shoulder.y) / 3.0
        body_center = Point2D(body_center_x, body_center_y)
        
        ball_to_body = cls.get_dist(ball, body_center)
        shoulder_width = max(1.0, cls.get_dist(offender.left_shoulder, offender.right_shoulder))
        protection_risk = max(0.0, (ball_to_body - shoulder_width) / shoulder_width)

        player_scale = max(1.0, cls.get_dist(offender.hip, offender.left_ankle))
        ground_y = (offender.left_ankle.y + offender.right_ankle.y) / 2.0
        ball_height = max(0.0, ground_y - ball.y)
        height_ratio = ball_height / (player_scale * 1.2)
        
        height_risk = 0.35 if height_ratio > 1.2 else 0.15 if height_ratio > 0.9 else 0.0

        # 💡 安全防護：若沒有有效防守者進入防禦半徑，防守相關風險直接歸零
        if defender is not None:
            dist_def_l = cls.get_dist(ball, defender.left_wrist)
            dist_def_r = cls.get_dist(ball, defender.right_wrist)
            closest_hand_dist = min(dist_def_l, dist_def_r)
            
            reach_limit = player_scale * 0.7
            reach_risk = max(0.0, (reach_limit - closest_hand_dist) / reach_limit) if closest_hand_dist < reach_limit else 0.0

            def_feet = Point2D((defender.left_ankle.x + defender.right_ankle.x)/2, (defender.left_ankle.y + defender.right_ankle.y)/2)
            ball_to_feet = cls.get_dist(ball, def_feet)
            block_limit = player_scale * 0.9
            blocking_risk = max(0.0, (block_limit - ball_to_feet) / block_limit) if ball_to_feet < block_limit else 0.0

            steal_prob = (protection_risk * 0.25) + (height_risk * 0.15) + (reach_risk * 0.35) + (blocking_risk * 0.25)
        else:
            closest_hand_dist = 0.0
            ball_to_feet = 0.0
            steal_prob = 0.0  # 無防守者威脅，被抄截率降為 0

        return {
            "steal_prob": min(0.99, max(0.01, steal_prob)),
            "ball_to_body": ball_to_body,
            "height_ratio": height_ratio,
            "defender_hand_dist": closest_hand_dist,
            "defender_foot_dist": ball_to_feet
        }

# ==========================================
# 3. 雙深度學習模型影片分析管線
# ==========================================
class DualModelBasketballAnalyzer:
    def __init__(self):
        print("正在初始化高精確度雙深度學習模型管線...")
        self.pose_model = YOLO("yolov8n-pose.pt")   
        self.object_model = YOLO("yolov8m.pt")      
        self.prob_history = []
        self.last_known_ball: Optional[Point2D] = None 

    def _extract_yolo_skeleton(self, keypoints, player_id: int) -> PlayerSkeleton:
        kp = keypoints.xy[0].cpu().numpy()
        return PlayerSkeleton(
            player_id=player_id,
            left_shoulder=Point2D(float(kp[5][0]), float(kp[5][1])),
            right_shoulder=Point2D(float(kp[6][0]), float(kp[6][1])),
            hip=Point2D(float((kp[11][0] + kp[12][0])/2), float((kp[11][1] + kp[12][1])/2)),
            left_wrist=Point2D(float(kp[9][0]), float(kp[9][1])),
            right_wrist=Point2D(float(kp[10][0]), float(kp[10][1])),
            left_ankle=Point2D(float(kp[15][0]), float(kp[15][1])),
            right_ankle=Point2D(float(kp[16][0]), float(kp[16][1]))
        )

    def analyze_video(self, video_path: str, output_path: str = "dual_model_output.mp4"):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened(): return

        width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        
        print(f" [智慧校正版] 開始解析影片: {video_path}")
        frame_idx = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            
            pose_results = self.pose_model.track(frame, persist=True, verbose=False, conf=0.25)
            obj_results = self.object_model(frame, verbose=False, conf=0.15)[0]
            
            detected_ball: Optional[Point2D] = None
            for box in obj_results.boxes:
                if int(box.cls[0]) == 32:  # 籃球
                    xyxy = box.xyxy[0].cpu().numpy()
                    detected_ball = Point2D((xyxy[0] + xyxy[2]) / 2.0, (xyxy[1] + xyxy[3]) / 2.0)
                    self.last_known_ball = detected_ball  
                    break

            steal_percentage = 0.0
            geo = {"ball_to_body": 0.0, "height_ratio": 0.0, "defender_hand_dist": 0.0, "defender_foot_dist": 0.0}

            if pose_results and pose_results[0].keypoints is not None and pose_results[0].boxes.id is not None:
                ids = pose_results[0].boxes.id.cpu().numpy().astype(int)
                keypoints_list = pose_results[0].keypoints
                players = [self._extract_yolo_skeleton(keypoints_list[idx], p_id) for idx, p_id in enumerate(ids)]

                # 「動態防禦半徑邏輯」
                if len(players) >= 2:
                    ref_ball = detected_ball if detected_ball is not None else self.last_known_ball
                    
                    if ref_ball is not None:
                        # 1. 找出離球最近的進攻者
                        offender = min(players, key=lambda p: min(UltimateBasketballEngine.get_dist(ref_ball, p.left_wrist), 
                                                                  UltimateBasketballEngine.get_dist(ref_ball, p.right_wrist)))
                        
                        # 2. 動態防禦半徑邏輯 (Dynamic Engagement Zone)
                        # 定義防禦半徑：以進攻者身長為基準 (1.5 倍身高)
                        player_height = UltimateBasketballEngine.get_dist(offender.hip, offender.left_ankle) * 2
                        MAX_DEFENSE_RADIUS = player_height * 1.5 
                        
                        remaining_players = [p for p in players if p.player_id != offender.player_id]
                        
                        # 過濾出在半徑內的對手
                        potential_defenders = [p for p in remaining_players if UltimateBasketballEngine.get_dist(offender.hip, p.hip) < MAX_DEFENSE_RADIUS]
                        
                        if potential_defenders:
                            # 如果有人在半徑內，選最近的一個為防守者
                            defender = min(potential_defenders, key=lambda p: UltimateBasketballEngine.get_dist(offender.hip, p.hip))
                        else:
                            # 沒有人在半徑內，不顯示防守者
                            defender = None 
                    else:
                        # 如果連最後已知球位都沒有，才暫時用 ID 排序保底
                        players = sorted(players, key=lambda p: p.player_id)
                        offender, defender = players[0], players[1]

                    # 防掉影保底：球掉幀時，鎖定進攻者的手腕位置
                    if detected_ball is None:
                        actual_ball = self.last_known_ball if self.last_known_ball is not None else offender.right_wrist
                    else:
                        actual_ball = detected_ball

                    # 數據計算
                    geo = UltimateBasketballEngine.analyze_real_analytics(actual_ball, offender, defender)
                    self.prob_history.append(geo["steal_prob"])
                    if len(self.prob_history) > 6: self.prob_history.pop(0)
                    steal_percentage = (sum(self.prob_history) / len(self.prob_history)) * 100

                    # --- 🎨 幾何繪線渲染 ---
                    # 1. 綠線：球到真正進攻者的身體中心
                    body_center_pos = (int((offender.hip.x + offender.left_shoulder.x)/2), int((offender.hip.y + offender.left_shoulder.y)/2))
                    cv2.line(frame, body_center_pos, (int(actual_ball.x), int(actual_ball.y)), (0, 255, 0), 2)
                    cv2.circle(frame, (int(offender.hip.x), int(offender.hip.y)), 6, (0, 255, 0), -1)
                    cv2.putText(frame, f"OFFENDER ID:{offender.player_id}", (int(offender.hip.x), int(offender.hip.y)-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    # 2. 紅線與防守者標註：只有在防守者成功進入防禦半徑時才進行繪製
                    if defender is not None:
                        dist_l = UltimateBasketballEngine.get_dist(actual_ball, defender.left_wrist)
                        dist_r = UltimateBasketballEngine.get_dist(actual_ball, defender.right_wrist)
                        def_hand_x = int(defender.left_wrist.x if dist_l < dist_r else defender.right_wrist.x)
                        def_hand_y = int(defender.left_wrist.y if dist_l < dist_r else defender.right_wrist.y)
                        cv2.line(frame, (def_hand_x, def_hand_y), (int(actual_ball.x), int(actual_ball.y)), (0, 0, 255), 2)
                        
                        cv2.circle(frame, (int(defender.hip.x), int(defender.hip.y)), 6, (0, 0, 255), -1)
                        cv2.putText(frame, f"DEFENDER ID:{defender.player_id}", (int(defender.hip.x), int(defender.hip.y)-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                    
                    # 3. 繪製真實籃球 (亮橘色圓圈)
                    cv2.circle(frame, (int(actual_ball.x), int(actual_ball.y)), 12, (0, 100, 255), -1)
                    cv2.circle(frame, (int(actual_ball.x), int(actual_ball.y)), 15, (255, 255, 255), 2)

            # --- 儀表板 UI 渲染 ---
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (520, 220), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            status_color = (0, 255, 0) if steal_percentage < 40 else (0, 165, 255) if steal_percentage < 75 else (0, 0, 255)
            cv2.putText(frame, "DUAL-MODEL REAL-BALL TRACKING", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"STEAL PROBABILITY: {steal_percentage:.1f}%", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)
            cv2.putText(frame, f" - Real Ball to Offender Body: {geo['ball_to_body']:.1f} px", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
            cv2.putText(frame, f" - Dribble Height Status    : {geo['height_ratio']:.2f} ratio", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
            
            # 儀表板安全顯示文字優化
            if geo['defender_hand_dist'] > 0:
                cv2.putText(frame, f" - Defender Hand to Ball Dist: {geo['defender_hand_dist']:.1f} px", (20, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
                cv2.putText(frame, f" - Defender Foot to Ball Dist: {geo['defender_foot_dist']:.1f} px", (20, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
            else:
                cv2.putText(frame, f" - Defender Hand to Ball Dist: Out of Range (Safe)", (20, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                cv2.putText(frame, f" - Defender Foot to Ball Dist: Out of Range (Safe)", (20, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            out.write(frame)
            cv2.imshow("Dual-Model Basketball Analytics System", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break

        cap.release()
        out.release()
        cv2.destroyAllWindows()
        print(f"🎉 分析完畢！檔案已儲存至: {output_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.withdraw()
    root.lift()
    root.attributes("-topmost", True)

    print(" 請選取你要分析的籃球影片...")
    chosen_video_path = filedialog.askopenfilename(parent=root, title="請選擇籃球影片", filetypes=[("MP4 影片", "*.mp4"), ("所有檔案", "*.*")])

    if chosen_video_path:
        final_output_path = os.path.join(os.path.dirname(chosen_video_path), f"calibrated_analyzed_{os.path.basename(chosen_video_path)}")
        analyzer = DualModelBasketballAnalyzer()
        analyzer.analyze_video(video_path=chosen_video_path, output_path=final_output_path)
    root.destroy()

 請選取你要分析的籃球影片...
正在初始化高精確度雙深度學習模型管線...
 [智慧校正版] 開始解析影片: C:/Users/wenra/basketball pyhon video/pro_fail_1.mp4
🎉 分析完畢！檔案已儲存至: C:/Users/wenra/basketball pyhon video\calibrated_analyzed_pro_fail_1.mp4
